# 1. Data Cleaning and Machine Learning — Parts A and B

Run the cells from top to bottom with **Run All**. Keep this file inside `instamart-copilot/notebooks/` and select a Python kernel with the project's requirements installed. The tested source modules contain the implementation; this notebook explains, runs and displays each stage without duplicating a second version of the algorithms.

All data is fictional and monetary values are INR. Outputs are text/tables, with no chart files. This notebook regenerates raw/clean data, SQLite and ML outputs. The two required supplied scripts remain unchanged. Run notebook 2 after this notebook. The command-line `run_project.py` remains the canonical complete reproduction route.


## 1.1 Locate the project and import libraries
The setup works when the kernel starts in the project root or its notebooks subfolder. It contains no machine-specific paths.


In [1]:
from pathlib import Path
import os, sys, json, runpy
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '2')
start = Path.cwd().resolve()
candidates = [start, *start.parents, start / 'instamart-copilot']
ROOT = next((p for p in candidates if (p / 'run_project.py').is_file() and (p / 'src').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the instamart-copilot project folder or its notebooks subfolder.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
OUT = ROOT / 'outputs'
def report(name):
    return json.loads((OUT / name).read_text(encoding='utf-8'))
print('Python:', sys.version.split()[0])
print('Project:', ROOT.name)


Python: 3.12.14
Project: instamart-copilot


## 1.2 Generate the supplied synthetic data
The original generator uses seed 42 and must produce exactly 13,084 raw order rows and 164 tickets.


In [2]:
_ = runpy.run_path(str(ROOT / 'generate_data.py'), run_name='__main__')
raw_customers = pd.read_csv(ROOT / 'data/raw/customers.csv', keep_default_na=False)
raw_orders = pd.read_csv(ROOT / 'data/raw/orders.csv', keep_default_na=False)
print('Customer preview:\n', raw_customers.head().to_string(index=False))
print('\nOrder preview:\n', raw_orders.head().to_string(index=False))
assert len(raw_customers) == 1200 and len(raw_orders) == 13084


generated: customers=1200 products=40 orders_raw=13084 tickets=164
Customer preview:
 customer_id      city signup_date age gender
      C0001   Chennai  2024-01-13  35   MALE
      C0002     DELHI  2024-02-17  31   MALE
      C0003 bengaluru  2025-02-18  18      F
      C0004 Bengaluru  2024-06-21  24 FEMALE
      C0005 Bengaluru  2024-10-01  25 FEMALE

Order preview:
   order_id customer_id product_id  quantity            order_ts    status  discount_pct order_amount delivery_minutes
ORD-108250       C0792       P004         2 2025-09-01 12:24:59 completed             0        310.0               17
ORD-101939       C0190       P003         1 2025-08-17 07:07:23 completed             5       232.75               21
ORD-108765       C0841       P033         1 2025-07-13 09:42:32 completed            10       1349.1               27
ORD-108861       C0852       P010         1 2025-07-04 11:23:24 completed             0         45.0               30
ORD-109150       C0877       P018    

## 1.3 Audit before cleaning
Inspect shapes, inferred types and missing counts/percentages. Empty CSV fields remain empty strings during the audit. Non-numeric planted amounts and invalid ages are counted separately.


In [3]:
from src.data_pipeline import audit, clean_data, validate, analyse
audit_report = audit()
print(pd.Series(audit_report['specific']).to_string())
for table_name in ['customers', 'products', 'orders']:
    info = audit_report[table_name]
    print('\n', table_name, 'shape:', info['shape'])
    details = pd.DataFrame({'dtype': info['dtypes'], 'missing': info['missing_count'], 'missing_pct': info['missing_pct']})
    print(details.to_string())


raw_rows               13084
exact_duplicates         119
pending                  656
distinct_orders        12309
non_numeric_amounts      120
negative_delivery        106
blank_delivery          2541
blank_age                 61
invalid_age                6
city_spellings            24
gender_spellings          10
blank_gender              30

 customers shape: [1200, 5]
              dtype  missing  missing_pct
customer_id  object        0     0.000000
city         object        0     0.000000
signup_date  object        0     0.000000
age          object       61     5.083333
gender       object       30     2.500000

 products shape: [40, 4]
               dtype  missing  missing_pct
product_id    object        0          0.0
product_name  object        0          0.0
category      object        0          0.0
price_inr      int64        0          0.0

 orders shape: [13084, 9]
                   dtype  missing  missing_pct
order_id          object        0     0.000000
customer_

## 1.4 Clean and persist the three tables

The cleaning implementation in `src/data_pipeline.py` applies these rules in order:
1. Strip/title-case city, normalize gender, invalidate ages outside 10–100, and impute age using the customer's city median.
2. Parse timestamps, remove exact duplicates, then keep the latest record per order ID.
3. Reconstruct missing amounts from product price, quantity and discount.
4. Clear negative delivery durations and impute missing completed durations using category medians. Cancelled durations stay missing.
5. Save the exact required CSVs and SQLite tables.

The following output reports the actual number of affected values, not just the planned rules.


In [4]:
customers, products, orders = clean_data()
print('Cleaning changes:\n', pd.Series(report('cleaning_changes.json')).to_string())
print('\nClean shapes:', customers.shape, products.shape, orders.shape)
print('\nFinal order statuses:\n', orders.status.value_counts().to_string())
print('\nCity counts:\n', customers.city.value_counts().to_string())
validate()


Cleaning changes:
 city_values_changed            476
gender_values_changed          950
ages_imputed                    67
exact_rows_removed             119
earlier_records_removed        656
amounts_reconstructed          112
negative_durations_cleared     105
completed_durations_imputed    774

Clean shapes: (1200, 5) (40, 4) (12309, 9)

Final order statuses:
 status
completed    11110
cancelled     1199

City counts:
 city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
ALL CHECKPOINTS PASSED


## 1.5 Seven SQL queries and pandas cross-checks
The queries exercise WHERE/HAVING, a three-table join, scalar subquery, LEFT JOIN, CTEs, ranking windows and cumulative revenue. The pandas pivot grand total must equal the SQL total.


In [5]:
analysis = analyse()
print((ROOT / 'sql/queries.sql').read_text(encoding='utf-8'))
for number in range(1, 8):
    result = pd.read_csv(OUT / f'query_{number}.csv')
    print(f'\nQ{number}: {len(result)} rows; first five:')
    print(result.head().to_string(index=False))
print('\nCity named aggregations:\n', pd.read_csv(OUT / 'city_summary.csv').to_string(index=False))
print('\nDistribution statistics:')
print(pd.Series({k:v for k,v in analysis.items() if k != 'top_customers'}).to_string())
print('\nCorrelation matrix:\n', pd.read_csv(OUT / 'correlations.csv').to_string(index=False))


-- Q1 WHERE then HAVING
SELECT customer_id, SUM(order_amount) AS total_spend FROM orders
WHERE status='completed' AND order_amount>500 GROUP BY customer_id HAVING SUM(order_amount)>8000;
-- Q2 three-table join
SELECT c.city,p.category,SUM(o.order_amount) AS revenue FROM orders o
JOIN customers c USING(customer_id) JOIN products p USING(product_id)
WHERE o.status='completed' GROUP BY c.city,p.category ORDER BY revenue DESC;
-- Q3 scalar subquery
SELECT * FROM products WHERE price_inr>(SELECT AVG(price_inr) FROM products);
-- Q4 LEFT JOIN
SELECT c.customer_id FROM customers c LEFT JOIN orders o
ON c.customer_id=o.customer_id AND o.status='completed' WHERE o.order_id IS NULL;
-- Q5 two CTEs
WITH spend AS (SELECT customer_id,SUM(order_amount) AS total_spend FROM orders WHERE status='completed' GROUP BY customer_id),
average AS (SELECT AVG(total_spend) AS average_spend FROM spend)
SELECT spend.*,average_spend FROM spend CROSS JOIN average WHERE total_spend>average_spend;
-- Q6 city rankings

### Reading the data
Completed spending is right-skewed: mean INR 269.91 exceeds median INR 135.00. The high-value tail includes plausible electronics purchases; an IQR outlier is not automatically a data error. The largest absolute correlation is spending with discount at about −0.095, a weak association that does not establish causation. The README contains all required findings and numerical checkpoints.


## 1.6 RFM, K-means and PCA
RFM uses completed orders as of 2025-10-01. Standardize recency/frequency/monetary, compare K=2…8, then choose K=4 for its elbow, silhouette and actionable profiles. Cluster IDs have no ordering; Premium is exactly the cluster with the largest mean spend.


In [6]:
from src.customer_models import rfm_and_segments, build_features, build_lapse_model
rfm, diagnostics, profiles, pca_ratios = rfm_and_segments()
print('RFM statistics:\n', rfm[['recency','frequency','monetary']].describe().to_string())
print('\nK-means comparison:\n', diagnostics.to_string(index=False))
print('\nSegment profiles:\n', profiles.to_string(index=False))
print('\nPCA explained variance:', pca_ratios, 'total:', pca_ratios.sum())


RFM statistics:
            recency    frequency      monetary
count  1175.000000  1175.000000   1175.000000
mean     33.558298     9.455319   2552.058553
std      38.433705     8.009971   2367.305104
min       0.000000     1.000000     12.000000
25%       6.000000     3.000000    505.450000
50%      19.000000     5.000000   2286.500000
75%      46.000000    17.000000   3751.525000
max     180.000000    33.000000  14766.550000

K-means comparison:
  k     inertia  silhouette
 2 1820.760553    0.464972
 3 1268.234587    0.445161
 4  743.071108    0.525544
 5  617.366636    0.480495
 6  503.170622    0.481872
 7  408.460454    0.416337
 8  366.503487    0.412329

Segment profiles:
  segment_id  size    recency  frequency    monetary segment_name
          0   436  28.862385   3.772936 1057.618234   Occasional
          1   138  24.115942   6.768116 7129.695652      Premium
          2   419   8.766110  19.403341 3410.020525     Frequent
          3   182 109.043956   2.203297  685.992582

## 1.7 Define a temporal lapse problem
Only customers with a completed order before 2025-09-01 enter the cohort. Lapsed=1 means no completed order during September. Model features aggregate pre-cutoff orders; whole-window segment labels are excluded. Encoders/scalers are fitted within training/CV pipelines, not on test rows. Caveat: the required Part A full-window delivery imputation indirectly uses September medians; the README discloses this residual leakage.


In [7]:
X, y = build_features()
print('Feature names:', list(X.columns))
print('Cohort:', len(X), 'Lapsed:', int(y.sum()), 'Prevalence:', round(y.mean(), 4))
print('\nFeature preview:\n', X.head().to_string())
print('\nMissing feature values:', int(X.isna().sum().sum()))
assert len(X) == 1156 and y.sum() == 443 and not X.isna().any().any()


Feature names: ['recency', 'frequency', 'monetary', 'avg_discount', 'avg_delivery', 'cancellation_rate', 'favourite_category', 'city', 'age', 'gender', 'tenure']
Cohort: 1156 Lapsed: 443 Prevalence: 0.3832

Feature preview:
              recency  frequency  monetary  avg_discount  avg_delivery  cancellation_rate favourite_category       city   age  gender  tenure
customer_id                                                                                                                                 
C0001             33          2    378.00     10.000000     28.000000           0.000000            Grocery    Chennai  35.0    male     597
C0002             22          6   3946.90      6.666667     21.916667           0.000000        Electronics      Delhi  31.0    male     562
C0003              1         20   3264.05     12.750000     24.200000           0.000000            Grocery  Bengaluru  18.0  female     195
C0004             42          3    272.25     11.666667     35.666667 

## 1.8 Tune, evaluate, balance and save models
This cell may take a few minutes. Logistic regression tunes C={0.01,0.1,1,10,100}; the 100-tree forest tunes depth={3,5,8} and minimum leaf={5,10,20}, both with five-fold F1 scoring. The model is selected by CV, then evaluated on untouched test rows. Oversampling duplicates only minority training rows. The selected model must reach test F1≥0.55 and its reloaded predictions must be identical.


In [8]:
ml = build_lapse_model()
metrics_table = []
for name in ['baseline', 'lr', 'rf', 'balanced_lr']:
    m = ml[name]
    metrics_table.append({'model':name, **{k:m[k] for k in ['precision','recall','f1','accuracy']}, 'confusion_matrix':m['confusion_matrix']})
print(pd.DataFrame(metrics_table).to_string(index=False))
for name in ['lr', 'rf']:
    print(name, 'best parameters:', ml[name]['best_params'], 'CV F1:', ml[name]['cv_f1'])
print('Selected:', ml['selected'], 'Reload identical:', ml['reload_identical'])
print('Training/full-cohort frequency means:', ml['frequency_scaler_mean'], ml['frequency_cohort_mean'])
print('Balanced training class counts:', ml['balanced_counts'])
assert ml[ml['selected']]['f1'] >= .55 and ml['reload_identical']


      model  precision   recall       f1  accuracy      confusion_matrix
   baseline   0.000000 0.000000 0.000000  0.616379   [[143, 0], [89, 0]]
         lr   0.644444 0.651685 0.648045  0.728448 [[111, 32], [31, 58]]
         rf   0.629213 0.629213 0.629213  0.715517 [[110, 33], [33, 56]]
balanced_lr   0.589147 0.853933 0.697248  0.715517  [[90, 53], [13, 76]]
lr best parameters: {'model__C': 100} CV F1: 0.650552962900789
rf best parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 5} CV F1: 0.6828774929815611
Selected: rf Reload identical: True
Training/full-cohort frequency means: 8.114718614718615 8.028546712802768
Balanced training class counts: {1: 570, 0: 570}


## 1.9 Interpret and inspect the retention list
Forest importance is predictive association, not causation. Logistic coefficients describe direction conditional on other encoded inputs. The Premium flag uses the full window including September: this retention list is illustrative and is not a leakage-free evaluation.


In [9]:
print('Forest top five:\n', pd.DataFrame(ml['rf_top_five'], columns=['feature','importance']).to_string(index=False))
print('\nLogistic coefficient signs:\n', pd.Series(ml['lr_coefficients']).to_string())
retention = pd.read_csv(OUT / 'retention_list.csv')
print('\nIllustrative retention list:\n', retention.to_string(index=False))
print('Actually lapsed among selected customers:', int(retention.actually_lapsed.sum()))


Forest top five:
                feature  importance
        num__frequency    0.317188
         num__monetary    0.260046
          num__recency    0.112350
num__cancellation_rate    0.072730
     num__avg_delivery    0.049232

Logistic coefficient signs:
 recency      0.039585
frequency   -1.698239
monetary     0.284573

Illustrative retention list:
 customer_id  lapse_probability  actually_lapsed
      C0060           0.532795                0
      C0267           0.500643                0
      C1112           0.490648                0
      C0133           0.486396                0
      C0167           0.479013                0
      C0862           0.477373                0
      C0942           0.473873                0
      C0802           0.469605                0
      C1008           0.467918                0
      C0930           0.459018                0
Actually lapsed among selected customers: 0


## 1.10 Enter another customer's history
Edit `new_customer` below and rerun this cell. Use five months of history ending before the scoring month. The result estimates no completed order in the following month. A never-ordered customer cannot be scored; age/city alone are insufficient. No training data or saved model is modified.


In [10]:
from predict_customer import score_customer
new_customer = {
    'customer_id': 'NEW001',
    'recency': 20, 'frequency': 6, 'monetary': 1800,
    'avg_discount': 10, 'avg_delivery': 28, 'cancellation_rate': 0.1,
    'favourite_category': 'Grocery', 'city': 'Chennai',
    'age': 30, 'gender': 'female', 'tenure': 300,
}
prediction = score_customer(new_customer)
print('Customer:', prediction['customer_id'])
print(f"Estimated lapse probability: {prediction['lapse_probability']:.1%}")
print(prediction['prediction'])
print('Educational synthetic-data model; this is not an observed outcome.')


Customer: NEW001
Estimated lapse probability: 34.9%
LIKELY TO REMAIN ACTIVE
Educational synthetic-data model; this is not an observed outcome.


## 1.11 Check Part A/B and prediction tests
The complete suite is run at the end of notebook 2. Here we check the foundation, models and new-customer input handling.


In [11]:
import unittest
sys.path.insert(0, str(ROOT / 'tests'))
from test_project import Foundation, Models
from test_prediction import CustomerPrediction
suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromTestCase(t) for t in [Foundation, Models, CustomerPrediction])
test_result = unittest.TextTestRunner(stream=sys.stdout, verbosity=1).run(suite)
assert test_result.wasSuccessful()
print('DATA CLEANING AND ML NOTEBOOK COMPLETE')


..........
----------------------------------------------------------------------
Ran 10 tests in 1.432s

OK
DATA CLEANING AND ML NOTEBOOK COMPLETE
